# MLAAD Shared Data and MFCC Foundation

This notebook is the authoritative shared data foundation for the MLAAD audio
deepfake experiments. It owns the dataset audit, corrected metadata parsing, the
frozen 70/15/15 split, the optional full raw MFCC cache build, the shared class
weights, and the educational MFCC walkthrough.

Model notebooks should load saved artifacts from this notebook and must not rescan
the full dataset, make their own train/validation/test split, or re-extract MFCCs
from WAV files.


## 1. Environment Setup


In [ ]:
# Purpose: Documents the packages used by this notebook without installing anything
# automatically. Uncomment the next line in Colab or a fresh environment if a package
# is missing.
# %pip install librosa soundfile scikit-learn seaborn joblib


## 2. Imports


In [ ]:
# Purpose: Imports the standard-library and third-party tools used by the dataset
# audit, frozen split, MFCC cache creation, saved tables, and educational plots.
import json
import os
import random
import warnings
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)


## 3. Configuration and Random Seeds


In [ ]:
# Purpose: Keeps all audio and MFCC settings in one visible place so an examiner can
# see exactly what every downstream model is based on.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5.0
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))
TARGET_SAMPLES = int(round(SAMPLE_RATE * FIXED_DURATION_SECONDS))
MFCC_CENTER = False
EXPECTED_FRAMES = 1 + max(0, TARGET_SAMPLES - N_FFT) // HOP_LENGTH

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
MAX_FILES_PER_CLASS = None

# Purpose: Full-dataset MFCC extraction can take a long time, so it is intentionally
# opt-in. Set this to True when you are ready to build or rebuild the shared cache.
RUN_FULL_SHARED_MFCC_EXTRACTION = False


## 4. Dataset and Output Paths


In [ ]:
# Purpose: Resolves the same project root whether the notebook is run locally, from a
# nested Model Variants folder, or inside Colab with a mounted Drive.
def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()

    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


def first_existing_path(candidates):
    # Purpose: Lets the same notebook run on local Windows paths and Colab-style paths
    # without hard-coding one person's machine layout. Empty environment variables are
    # skipped because Path("") would otherwise mean the current working directory.
    expanded = []
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in candidates:
        if candidate in [None, ""]:
            continue
        expanded.append(Path(candidate).expanduser())
    # Purpose: Iterates over this collection to build the next table, feature set, or experiment result
    # Purpose: consistently.
    for path in expanded:
        if path.exists():
            return path
    return expanded[0]


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
DATA_ROOT = PROJECT_ROOT / "Data"

SYNTHETIC_AUDIO_DIR = first_existing_path([
    os.environ.get("MLAAD_SYNTHETIC_AUDIO_DIR", ""),
    DATA_ROOT / "MLAAD_10pct" / "fake",
    DATA_ROOT / "Unprocessed" / "MLAAD_10pct" / "fake",
    DATA_ROOT / "MLAAD_10pct",
    PROJECT_ROOT / "MLAAD_10pct" / "fake",
    PROJECT_ROOT / "MLAAD_10pct",
    PROJECT_ROOT / "Datasets" / "MLAAD_10pct",
])
BONA_FIDE_AUDIO_DIR = first_existing_path([
    os.environ.get("MLAAD_BONA_FIDE_AUDIO_DIR", ""),
    DATA_ROOT / "genuine_audio",
    DATA_ROOT / "bona_fide",
    DATA_ROOT / "Unprocessed" / "genuine_audio",
    PROJECT_ROOT / "genuine_audio",
    PROJECT_ROOT / "bona_fide",
    PROJECT_ROOT / "Datasets" / "M_AILABS_bona_fide_subset",
])

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
MFCC_CACHE_DIR = OUTPUT_DIR / "mfcc_cache"
CLASS_WEIGHT_PATH = OUTPUT_DIR / "class_weights.json"

# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, MANIFESTS_DIR, MFCC_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Synthetic audio directory:", SYNTHETIC_AUDIO_DIR)
print("Bona-fide audio directory:", BONA_FIDE_AUDIO_DIR)
print("Shared manifest directory:", MANIFESTS_DIR)
print("Shared raw MFCC cache directory:", MFCC_CACHE_DIR)


## 5. Corrected Dataset Metadata Scanner


In [ ]:
# Purpose: Builds one corrected manifest for the whole project. Synthetic MLAAD paths
# usually look like fake/<language>/<tts_generator>/<file>, so language and generator
# must not be collapsed into path.parent.name.
MANIFEST_COLUMNS = [
    "path",
    "relative_path",
    "label",
    "class_name",
    "language",
    "tts_generator",
    "file_size_mb",
    "metadata_warning",
]


def parse_synthetic_metadata(path, root_dir):
    # Purpose: Extracts synthetic-language and generator labels from the dataset folder structure.
    relative_parts = path.relative_to(root_dir).parts
    warning = ""
    if relative_parts and relative_parts[0].lower() == "fake" and len(relative_parts) >= 3:
        language = relative_parts[1]
        tts_generator = relative_parts[2]
    elif len(relative_parts) >= 2:
        language = relative_parts[0]
        tts_generator = relative_parts[1]
    else:
        language = "unknown"
        tts_generator = "unknown"
        warning = "synthetic language/generator not derivable from path"
    return language, tts_generator, warning


def parse_bona_fide_metadata(path, root_dir):
    # Purpose: Extracts bona-fide metadata from the dataset folder structure when it is available.
    relative_parts = path.relative_to(root_dir).parts
    warning = ""
    language = relative_parts[0] if len(relative_parts) >= 2 else "unknown"
    if language == "unknown":
        warning = "bona_fide language not reliably derivable from path"
    return language, "bona_fide", warning


def scan_audio_files(root_dir, label, class_name, source_kind):
    # Purpose: Scans a dataset folder and builds one metadata row for each supported audio file.
    root_dir = Path(root_dir)
    rows = []
    if not root_dir.exists():
        print(f"Dataset directory does not exist, so no files were scanned: {root_dir}")
        return pd.DataFrame(rows, columns=MANIFEST_COLUMNS)

    # Purpose: Iterates over this collection to build the next table, feature set, or experiment result
    # Purpose: consistently.
    for path in sorted(root_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue
        if source_kind == "synthetic":
            language, tts_generator, warning = parse_synthetic_metadata(path, root_dir)
        else:
            language, tts_generator, warning = parse_bona_fide_metadata(path, root_dir)

        rows.append(
            {
                "path": str(path.resolve()),
                "relative_path": str(path.relative_to(root_dir)),
                "label": int(label),
                "class_name": class_name,
                "language": language,
                "tts_generator": tts_generator,
                "file_size_mb": path.stat().st_size / (1024 * 1024),
                "metadata_warning": warning,
            }
        )
    return pd.DataFrame(rows, columns=MANIFEST_COLUMNS)


## 6. Dataset Scan and Audit Tables


In [ ]:
# Purpose: Scans each class once, optionally caps rows only for a quick development
# check, and saves central audit tables instead of duplicating this in model notebooks.
synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, 1, CLASS_NAMES[1], "synthetic")
bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, 0, CLASS_NAMES[0], "bona_fide")

if MAX_FILES_PER_CLASS is not None:
    synthetic_manifest = synthetic_manifest.sample(
        n=min(MAX_FILES_PER_CLASS, len(synthetic_manifest)),
        random_state=RANDOM_STATE,
    ).sort_values("path")
    bona_fide_manifest = bona_fide_manifest.sample(
        n=min(MAX_FILES_PER_CLASS, len(bona_fide_manifest)),
        random_state=RANDOM_STATE,
    ).sort_values("path")

manifest = pd.concat([bona_fide_manifest, synthetic_manifest], ignore_index=True)
manifest = manifest.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

class_counts = manifest["class_name"].value_counts().rename_axis("class_name").reset_index(name="recordings")
language_counts = manifest["language"].value_counts().rename_axis("language").reset_index(name="recordings")
class_language_counts = (
    manifest.groupby(["class_name", "language"]).size().reset_index(name="recordings")
)
tts_generator_counts = (
    manifest.loc[manifest["label"] == 1, "tts_generator"]
    .value_counts()
    .rename_axis("tts_generator")
    .reset_index(name="synthetic_recordings")
)

# Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same outputs.
manifest.to_csv(TABLES_DIR / "full_corrected_manifest.csv", index=False)
class_counts.to_csv(TABLES_DIR / "class_counts.csv", index=False)
language_counts.to_csv(TABLES_DIR / "language_counts.csv", index=False)
class_language_counts.to_csv(TABLES_DIR / "class_language_counts.csv", index=False)
tts_generator_counts.to_csv(TABLES_DIR / "synthetic_tts_generator_counts.csv", index=False)

print("Total recordings:", len(manifest))
display(class_counts)
display(language_counts.head(20))
display(tts_generator_counts.head(20))


## 7. Audio File Quality Checks


In [ ]:
# Purpose: Reads lightweight physical audio metadata from disk without loading every
# full waveform into memory.
def read_audio_info(path):
    # Purpose: Keeps the read_audio_info helper isolated so later notebook cells can call it consistently.
    try:
        info = sf.info(path)
        duration_seconds = float(info.frames) / float(info.samplerate) if info.samplerate else np.nan
        return {
            "path": str(path),
            "audio_exists": True,
            "native_sample_rate": int(info.samplerate),
            "channels": int(info.channels),
            "frames": int(info.frames),
            "duration_seconds": duration_seconds,
            "audio_info_error": "",
        }
    except Exception as exc:
        return {
            "path": str(path),
            "audio_exists": Path(path).exists(),
            "native_sample_rate": np.nan,
            "channels": np.nan,
            "frames": np.nan,
            "duration_seconds": np.nan,
            "audio_info_error": str(exc),
        }


audio_info_columns = ["path", "audio_exists", "native_sample_rate", "channels", "frames", "duration_seconds", "audio_info_error"]
audio_info_df = pd.DataFrame([read_audio_info(path) for path in manifest["path"]], columns=audio_info_columns)
manifest_with_audio_info = manifest.merge(audio_info_df, on="path", how="left", validate="one_to_one")
duration_summary = (
    manifest_with_audio_info.groupby("class_name")["duration_seconds"]
    .describe()
    .reset_index()
)
sample_rate_summary = (
    manifest_with_audio_info.groupby(["class_name", "native_sample_rate"])
    .size()
    .reset_index(name="recordings")
    .sort_values(["class_name", "recordings"], ascending=[True, False])
)
quality_checks = pd.DataFrame(
    [
        {"check": "duplicate_paths", "value": int(manifest["path"].duplicated().sum())},
        {"check": "missing_files", "value": int((~manifest["path"].map(lambda p: Path(p).exists())).sum())},
        {"check": "audio_info_errors", "value": int((audio_info_df["audio_info_error"] != "").sum())},
        {"check": "metadata_warnings", "value": int((manifest["metadata_warning"].fillna("") != "").sum())},
    ]
)

# Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same outputs.
manifest_with_audio_info.to_csv(TABLES_DIR / "manifest_with_audio_info.csv", index=False)
duration_summary.to_csv(TABLES_DIR / "duration_summary_by_class.csv", index=False)
sample_rate_summary.to_csv(TABLES_DIR / "native_sample_rate_by_class.csv", index=False)
quality_checks.to_csv(TABLES_DIR / "data_quality_checks.csv", index=False)

display(quality_checks)
display(duration_summary)
display(sample_rate_summary.head(20))


## 8. Audit Plots


In [ ]:
# Purpose: Saves quick examiner-friendly figures for the central dataset audit.
if not class_counts.empty:
    plt.figure(figsize=(5, 4))
    sns.barplot(data=class_counts, x="class_name", y="recordings")
    plt.title("Recordings by class")
    plt.xlabel("Class")
    plt.ylabel("Recordings")
    plt.tight_layout()
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    plt.savefig(FIGURES_DIR / "shared_class_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()

if not manifest_with_audio_info.empty:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=manifest_with_audio_info, x="duration_seconds", hue="class_name", bins=40)
    plt.title("Audio duration distribution")
    plt.xlabel("Duration (seconds)")
    plt.ylabel("Recordings")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shared_duration_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()


## 9. Canonical Frozen 70/15/15 Split


In [ ]:
# Purpose: Creates or reloads the one authoritative train/validation/test split. The
# split files are reused on later executions instead of being regenerated each run.
train_manifest_path = MANIFESTS_DIR / "train_manifest.csv"
validation_manifest_path = MANIFESTS_DIR / "validation_manifest.csv"
test_manifest_path = MANIFESTS_DIR / "test_manifest.csv"
split_config_path = MANIFESTS_DIR / "split_config.json"

GROUP_KEY = None

if all(path.exists() for path in [train_manifest_path, validation_manifest_path, test_manifest_path, split_config_path]):
    train_manifest = pd.read_csv(train_manifest_path)
    validation_manifest = pd.read_csv(validation_manifest_path)
    test_manifest = pd.read_csv(test_manifest_path)
    with open(split_config_path, "r", encoding="utf-8") as f:
        split_config = json.load(f)
    print("Loaded existing canonical split from:", MANIFESTS_DIR)
else:
    if manifest.empty:
        raise RuntimeError("No recordings were found, so the canonical split cannot be created.")
    if GROUP_KEY is None:
        print("No reliable group/source identifier was derived. Using reproducible label-stratified splitting.")
    train_manifest, remaining_manifest = train_test_split(
        manifest,
        train_size=0.70,
        random_state=RANDOM_STATE,
        stratify=manifest["label"],
    )
    validation_manifest, test_manifest = train_test_split(
        remaining_manifest,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=remaining_manifest["label"],
    )
    train_manifest = train_manifest.reset_index(drop=True)
    validation_manifest = validation_manifest.reset_index(drop=True)
    test_manifest = test_manifest.reset_index(drop=True)

    split_config = {
        "random_state": RANDOM_STATE,
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
        "group_key_used_or_null": GROUP_KEY,
        "creation_method": "label-stratified train_test_split; old model-local pilot splits retired",
        "sample_rate": SAMPLE_RATE,
        "fixed_duration_seconds": FIXED_DURATION_SECONDS,
        "n_mfcc": N_MFCC,
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "win_length": WIN_LENGTH,
        "expected_frames": EXPECTED_FRAMES,
    }
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    train_manifest.to_csv(train_manifest_path, index=False)
    validation_manifest.to_csv(validation_manifest_path, index=False)
    test_manifest.to_csv(test_manifest_path, index=False)
    with open(split_config_path, "w", encoding="utf-8") as f:
        json.dump(split_config, f, indent=2)

path_sets = {
    "train": set(train_manifest["path"]),
    "validation": set(validation_manifest["path"]),
    "test": set(test_manifest["path"]),
}
overlaps = {
    "train_validation_path_overlap": len(path_sets["train"] & path_sets["validation"]),
    "train_test_path_overlap": len(path_sets["train"] & path_sets["test"]),
    "validation_test_path_overlap": len(path_sets["validation"] & path_sets["test"]),
}
if any(overlaps.values()):
    raise RuntimeError(f"Canonical split path overlap detected: {overlaps}")

split_summary = (
    pd.concat(
        [
            train_manifest.assign(split="train"),
            validation_manifest.assign(split="validation"),
            test_manifest.assign(split="test"),
        ],
        ignore_index=True,
    )
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="recordings")
)
split_summary.to_csv(TABLES_DIR / "canonical_split_class_counts.csv", index=False)
display(split_summary)
print("Canonical split config:", split_config)
print("Path overlap checks:", overlaps)


## 10. Shared Raw MFCC Matrix Cache


In [ ]:
# Purpose: Defines the one expensive MFCC extraction function used by all neural model
# branches. MLP, CNN and LSTM later reshape or aggregate this same source matrix.
def load_audio_fixed(path):
    # Purpose: Loads one audio file, converts it to mono, resamples it, and forces a fixed duration.
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(audio) > TARGET_SAMPLES:
        audio = audio[:TARGET_SAMPLES]
    elif len(audio) < TARGET_SAMPLES:
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
    return audio.astype(np.float32)


def pad_or_trim_mfcc(mfcc_matrix):
    # Purpose: Keeps the pad_or_trim_mfcc helper isolated so later notebook cells can call it consistently.
    if mfcc_matrix.shape[1] > EXPECTED_FRAMES:
        mfcc_matrix = mfcc_matrix[:, :EXPECTED_FRAMES]
    elif mfcc_matrix.shape[1] < EXPECTED_FRAMES:
        pad_width = EXPECTED_FRAMES - mfcc_matrix.shape[1]
        mfcc_matrix = np.pad(mfcc_matrix, ((0, 0), (0, pad_width)))
    return mfcc_matrix.astype(np.float32)


def extract_raw_mfcc_matrix(path):
    # Purpose: consistently.
    audio = load_audio_fixed(path)
    mfcc_matrix = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window="hann",
        center=MFCC_CENTER,
    )
    return pad_or_trim_mfcc(mfcc_matrix)


def build_mfcc_split_cache(split_manifest, split_name):
    # Purpose: consistently.
    matrices = []
    labels = []
    skipped = []
    # Purpose: Processes one manifest row at a time so every audio file receives one feature vector and label.
    for row_index, row in split_manifest.reset_index(drop=True).iterrows():
        try:
            matrices.append(extract_raw_mfcc_matrix(row["path"]))
            labels.append(int(row["label"]))
        except Exception as exc:
            skipped.append({"split": split_name, "row_index": int(row_index), "path": row["path"], "error": str(exc)})
    if skipped:
        skipped_path = MFCC_CACHE_DIR / "feature_extraction_skipped.csv"
        # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
        # Purpose: outputs.
        pd.DataFrame(skipped).to_csv(skipped_path, index=False)
        raise RuntimeError(f"MFCC extraction failed for {len(skipped)} files. See {skipped_path}")
    return np.stack(matrices).astype(np.float32), np.asarray(labels, dtype=np.int64)


cache_files = {
    "train": (MFCC_CACHE_DIR / "X_train_mfcc.npy", MFCC_CACHE_DIR / "y_train.npy", MFCC_CACHE_DIR / "train_metadata.csv"),
    "validation": (MFCC_CACHE_DIR / "X_validation_mfcc.npy", MFCC_CACHE_DIR / "y_validation.npy", MFCC_CACHE_DIR / "validation_metadata.csv"),
    "test": (MFCC_CACHE_DIR / "X_test_mfcc.npy", MFCC_CACHE_DIR / "y_test.npy", MFCC_CACHE_DIR / "test_metadata.csv"),
}
mfcc_config_path = MFCC_CACHE_DIR / "mfcc_config.json"
shared_cache_available = all(path.exists() for files in cache_files.values() for path in files) and mfcc_config_path.exists()

if shared_cache_available:
    X_train_mfcc = np.load(cache_files["train"][0])
    y_train = np.load(cache_files["train"][1])
    train_metadata = pd.read_csv(cache_files["train"][2])
    X_validation_mfcc = np.load(cache_files["validation"][0])
    y_validation = np.load(cache_files["validation"][1])
    validation_metadata = pd.read_csv(cache_files["validation"][2])
    X_test_mfcc = np.load(cache_files["test"][0])
    y_test = np.load(cache_files["test"][1])
    test_metadata = pd.read_csv(cache_files["test"][2])
    with open(mfcc_config_path, "r", encoding="utf-8") as f:
        mfcc_config = json.load(f)
    print("Loaded existing shared raw MFCC cache from:", MFCC_CACHE_DIR)
elif RUN_FULL_SHARED_MFCC_EXTRACTION:
    X_train_mfcc, y_train = build_mfcc_split_cache(train_manifest, "train")
    X_validation_mfcc, y_validation = build_mfcc_split_cache(validation_manifest, "validation")
    X_test_mfcc, y_test = build_mfcc_split_cache(test_manifest, "test")
    train_metadata = train_manifest.reset_index(drop=True).copy()
    validation_metadata = validation_manifest.reset_index(drop=True).copy()
    test_metadata = test_manifest.reset_index(drop=True).copy()

    np.save(cache_files["train"][0], X_train_mfcc)
    np.save(cache_files["train"][1], y_train)
    train_metadata.to_csv(cache_files["train"][2], index=False)
    np.save(cache_files["validation"][0], X_validation_mfcc)
    np.save(cache_files["validation"][1], y_validation)
    validation_metadata.to_csv(cache_files["validation"][2], index=False)
    np.save(cache_files["test"][0], X_test_mfcc)
    np.save(cache_files["test"][1], y_test)
    test_metadata.to_csv(cache_files["test"][2], index=False)

    mfcc_config = {
        "source": "shared raw MFCC matrix cache",
        "sample_rate": SAMPLE_RATE,
        "fixed_duration_seconds": FIXED_DURATION_SECONDS,
        "target_samples": TARGET_SAMPLES,
        "n_mfcc": N_MFCC,
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "win_length": WIN_LENGTH,
        "center": MFCC_CENTER,
        "expected_frames": EXPECTED_FRAMES,
        "matrix_shape_per_recording": [N_MFCC, EXPECTED_FRAMES],
        "dtype": "float32",
    }
    with open(mfcc_config_path, "w", encoding="utf-8") as f:
        json.dump(mfcc_config, f, indent=2)
    shared_cache_available = True
    print("Built and saved shared raw MFCC cache:", MFCC_CACHE_DIR)
else:
    print("Shared raw MFCC cache is missing and RUN_FULL_SHARED_MFCC_EXTRACTION is False.")
    print("Set RUN_FULL_SHARED_MFCC_EXTRACTION = True in the configuration cell to build it intentionally.")
    X_train_mfcc = X_validation_mfcc = X_test_mfcc = None
    y_train = y_validation = y_test = None
    train_metadata = validation_metadata = test_metadata = None
    mfcc_config = None


## 11. Raw MFCC Cache Row-Order Checks


In [ ]:
# Purpose: Proves that any loaded/built MFCC arrays preserve the canonical manifest
# row order and label order exactly.
if shared_cache_available:
    cache_checks = {
        "train_rows_match_manifest": len(X_train_mfcc) == len(train_manifest),
        "validation_rows_match_manifest": len(X_validation_mfcc) == len(validation_manifest),
        "test_rows_match_manifest": len(X_test_mfcc) == len(test_manifest),
        "train_labels_align": np.array_equal(y_train, train_manifest["label"].to_numpy(dtype=np.int64)),
        "validation_labels_align": np.array_equal(y_validation, validation_manifest["label"].to_numpy(dtype=np.int64)),
        "test_labels_align": np.array_equal(y_test, test_manifest["label"].to_numpy(dtype=np.int64)),
        "raw_mfcc_shape": list(X_train_mfcc.shape[1:]),
    }
    if not all(value is True or isinstance(value, list) for value in cache_checks.values()):
        raise RuntimeError(f"Shared MFCC cache alignment check failed: {cache_checks}")
    display(pd.DataFrame(list(cache_checks.items()), columns=["check", "value"]))
else:
    print("Row-order checks skipped because the shared raw MFCC cache has not been built or loaded yet.")


## 12. Shared Class Weights


In [ ]:
# Purpose: Computes class weights once from the canonical training labels only. All
# MLP/CNN/LSTM experiments load this JSON instead of recalculating local weights.
def balanced_class_weights(y):
    # Purpose: consistently.
    labels, counts = np.unique(np.asarray(y, dtype=np.int64), return_counts=True)
    total = counts.sum()
    n_classes = len(labels)
    return {int(label): float(total / (n_classes * count)) for label, count in zip(labels, counts)}


canonical_y_train = train_manifest["label"].to_numpy(dtype=np.int64)
shared_class_weights = balanced_class_weights(canonical_y_train)
class_weight_payload = {
    "class_weights": {str(label): weight for label, weight in shared_class_weights.items()},
    "source": "canonical train_manifest labels only",
    "class_counts": {str(k): int(v) for k, v in train_manifest["label"].value_counts().sort_index().items()},
    "split_config": str(split_config_path),
}
with open(CLASS_WEIGHT_PATH, "w", encoding="utf-8") as f:
    json.dump(class_weight_payload, f, indent=2)

print("Saved shared class weights:", CLASS_WEIGHT_PATH)
display(pd.DataFrame(list(shared_class_weights.items()), columns=["label", "class_weight"]))


## MFCC Step-by-Step Visual Explanation


### Step 1: Select One Example Audio File


In [ ]:
# Purpose: Selects one readable training audio file for the educational walkthrough.
# Prefer a bona-fide example if one exists, otherwise use the first valid training row.
example_row = None
example_load_error = None

# Purpose: Iterates over this collection to build the next table, feature set, or experiment result
# Purpose: consistently.
for preferred_label in [0, 1]:
    candidates = train_manifest[train_manifest["label"] == preferred_label]
    # Purpose: Processes one manifest row at a time so every audio file receives one feature vector and label.
    for _, candidate in candidates.iterrows():
        candidate_path = Path(candidate["path"])
        if not candidate_path.exists():
            continue
        try:
            _ = sf.info(candidate_path)
            example_row = candidate
            break
        except Exception as exc:
            example_load_error = str(exc)
    if example_row is not None:
        break

if example_row is None:
    selected_audio_path = None
    print("No readable example audio file was found for the MFCC walkthrough.")
    if example_load_error:
        print("Last loading error:", example_load_error)
else:
    selected_audio_path = Path(example_row["path"])
    selected_example_df = pd.DataFrame(
        [
            {
                "path": str(selected_audio_path),
                "label": int(example_row["label"]),
                "class_name": example_row["class_name"],
                "language": example_row["language"],
                "tts_generator": example_row["tts_generator"],
            }
        ]
    )
    display(selected_example_df)


### Step 2: Raw Waveform


In [ ]:
# Purpose: Loads the selected WAV-like file and plots the raw amplitude over time.
if selected_audio_path is None:
    raw_audio = None
    raw_sample_rate = None
    print("Skipping raw waveform plot because no example audio was selected.")
else:
    try:
        raw_audio, raw_sample_rate = librosa.load(selected_audio_path, sr=None, mono=True)
        plt.figure(figsize=(10, 3))
        librosa.display.waveshow(raw_audio, sr=raw_sample_rate)
        plt.title("Step 2 - Raw waveform")
        plt.xlabel("Time (seconds)")
        plt.ylabel("Amplitude")
        plt.tight_layout()
        # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
        # Purpose: outputs.
        plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_02_raw_waveform.png", dpi=300, bbox_inches="tight")
        plt.show()
    except Exception as exc:
        raw_audio = None
        raw_sample_rate = None
        print(f"Could not load selected example audio: {selected_audio_path}")
        print("Error:", exc)


### Step 3: Fixed-Length Audio


In [ ]:
# Purpose: Shows the exact fixed-length waveform that later MFCC extraction receives.
if selected_audio_path is None:
    fixed_audio = None
    print("Skipping fixed-length plot because no example audio was selected.")
else:
    try:
        fixed_audio = load_audio_fixed(selected_audio_path)
        plt.figure(figsize=(10, 3))
        librosa.display.waveshow(fixed_audio, sr=SAMPLE_RATE)
        plt.title(f"Step 3 - Fixed-length waveform ({FIXED_DURATION_SECONDS:.1f} seconds)")
        plt.xlabel("Time (seconds)")
        plt.ylabel("Amplitude")
        plt.tight_layout()
        # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
        # Purpose: outputs.
        plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_03_fixed_length_waveform.png", dpi=300, bbox_inches="tight")
        plt.show()
    except Exception as exc:
        fixed_audio = None
        print("Could not standardise selected example audio.")
        print("Error:", exc)


### Step 4: Windowing


In [ ]:
# Purpose: Takes one short audio frame and applies a Hann window. Windowing prepares
# short audio frames for FFT analysis by tapering the frame edges.
if fixed_audio is None:
    raw_frame = None
    windowed_frame = None
    print("Skipping windowing plot because fixed-length audio is not available.")
else:
    frame_start = max(0, min(len(fixed_audio) // 2, len(fixed_audio) - WIN_LENGTH))
    raw_frame = fixed_audio[frame_start : frame_start + WIN_LENGTH]
    if len(raw_frame) < WIN_LENGTH:
        raw_frame = np.pad(raw_frame, (0, WIN_LENGTH - len(raw_frame)))
    hann_window = np.hanning(WIN_LENGTH).astype(np.float32)
    windowed_frame = raw_frame * hann_window
    frame_time_ms = np.arange(WIN_LENGTH) / SAMPLE_RATE * 1000

    plt.figure(figsize=(10, 4))
    plt.plot(frame_time_ms, raw_frame, label="raw frame", alpha=0.8)
    plt.plot(frame_time_ms, windowed_frame, label="Hann-windowed frame", alpha=0.8)
    plt.title("Step 4 - Raw frame vs Hann-windowed frame")
    plt.xlabel("Frame time (ms)")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.tight_layout()
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_04_windowing.png", dpi=300, bbox_inches="tight")
    plt.show()


### Step 5: Frequency Spectrum


In [ ]:
# Purpose: Uses the FFT to show which frequencies are present in the windowed frame.
if windowed_frame is None:
    print("Skipping frequency spectrum because no windowed frame is available.")
else:
    spectrum = np.fft.rfft(windowed_frame, n=N_FFT)
    magnitude = np.abs(spectrum)
    frequencies = np.fft.rfftfreq(N_FFT, d=1.0 / SAMPLE_RATE)

    plt.figure(figsize=(10, 4))
    plt.plot(frequencies, magnitude)
    plt.title("Step 5 - FFT magnitude spectrum")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.tight_layout()
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_05_frequency_spectrum.png", dpi=300, bbox_inches="tight")
    plt.show()


### Step 6: Mel Triangular Filterbank


In [ ]:
# Purpose: Shows the triangular Mel filters that group FFT bins into perceptual
# frequency bands before the log and cepstral steps.
mel_filterbank = librosa.filters.mel(
    sr=SAMPLE_RATE,
    n_fft=N_FFT,
    n_mels=N_MFCC,
    fmin=0.0,
    fmax=SAMPLE_RATE / 2,
)
plt.figure(figsize=(10, 4))
librosa.display.specshow(mel_filterbank, x_axis="linear", sr=SAMPLE_RATE)
plt.title("Step 6 - Mel triangular filterbank")
plt.xlabel("Frequency bin")
plt.ylabel("Mel filter")
plt.colorbar(label="Weight")
plt.tight_layout()
# Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same outputs.
plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_06_mel_filterbank.png", dpi=300, bbox_inches="tight")
plt.show()


### Step 7: Mel Spectrogram


In [ ]:
# Purpose: Converts the fixed-length waveform into Mel-band energy over time.
if fixed_audio is None:
    mel_spectrogram = None
    print("Skipping Mel spectrogram because fixed-length audio is not available.")
else:
    mel_spectrogram = librosa.feature.melspectrogram(
        y=fixed_audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window="hann",
        center=MFCC_CENTER,
        n_mels=N_MFCC,
        power=2.0,
    )
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_spectrogram, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel")
    plt.title("Step 7 - Mel spectrogram")
    plt.colorbar(label="Power")
    plt.tight_layout()
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_07_mel_spectrogram.png", dpi=300, bbox_inches="tight")
    plt.show()


### Step 8: Log-Mel Spectrogram


In [ ]:
# Purpose: Converts Mel power to decibels. MFCCs are computed from this log-energy
# style representation before the DCT compacts it into cepstral coefficients.
if mel_spectrogram is None:
    log_mel_spectrogram = None
    print("Skipping log-Mel spectrogram because Mel spectrogram is not available.")
else:
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(log_mel_spectrogram, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel")
    plt.title("Step 8 - Log-Mel spectrogram")
    plt.colorbar(label="dB")
    plt.tight_layout()
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_08_log_mel_spectrogram.png", dpi=300, bbox_inches="tight")
    plt.show()


### Step 9: MFCC Heatmap


In [ ]:
# Purpose: Computes the MFCC matrix with the same function used by the shared cache.
if selected_audio_path is None:
    example_mfcc_matrix = None
    print("Skipping MFCC heatmap because no example audio was selected.")
else:
    try:
        example_mfcc_matrix = extract_raw_mfcc_matrix(selected_audio_path)
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(example_mfcc_matrix, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time")
        plt.title("Step 9 - MFCC matrix heatmap")
        plt.ylabel("MFCC coefficient")
        plt.colorbar(label="Coefficient value")
        plt.tight_layout()
        # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
        # Purpose: outputs.
        plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_09_mfcc_heatmap.png", dpi=300, bbox_inches="tight")
        plt.show()
        print("MFCC matrix shape:", example_mfcc_matrix.shape)
    except Exception as exc:
        example_mfcc_matrix = None
        print("Could not compute MFCCs for the selected example.")
        print("Error:", exc)


### Step 10: MFCC Mean and Standard Deviation Aggregation


In [ ]:
# Purpose: Shows the fixed-vector branch used by MLP/SVM-style models. This compresses
# the 2D MFCC matrix into 40 coefficient means plus 40 coefficient standard deviations.
# CNN and LSTM do not consume this 80-D vector directly.
if example_mfcc_matrix is None:
    print("Skipping MFCC aggregation because no MFCC matrix is available.")
else:
    mfcc_means = example_mfcc_matrix.mean(axis=1)
    mfcc_stds = example_mfcc_matrix.std(axis=1)
    example_feature_vector = np.concatenate([mfcc_means, mfcc_stds]).astype(np.float32)

    coefficient_index = np.arange(1, N_MFCC + 1)
    plt.figure(figsize=(10, 4))
    plt.bar(coefficient_index, mfcc_means)
    plt.title("Step 10a - MFCC coefficient means")
    plt.xlabel("MFCC coefficient")
    plt.ylabel("Mean value")
    plt.tight_layout()
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_10a_mfcc_means.png", dpi=300, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.bar(coefficient_index, mfcc_stds)
    plt.title("Step 10b - MFCC coefficient standard deviations")
    plt.xlabel("MFCC coefficient")
    plt.ylabel("Standard deviation")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "mfcc_walkthrough_step_10b_mfcc_stds.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("Final fixed-vector feature shape:", example_feature_vector.shape)


Raw WAV -> fixed-length waveform -> windowed frames -> frequency spectrum ->
Mel filterbank -> log-Mel spectrogram -> MFCCs -> mean/std feature vector.


## How the Same MFCC Matrix Feeds Different Models

```text
                      MFCC matrix
                         40 x T
                           |
           +---------------+---------------+
           |               |               |
          MLP             CNN             LSTM
           |               |               |
     mean + std       retain 2D map    transpose/retain
           |            + channel       time sequence
           |               |               |
        80-D          40 x T x 1        T x 40
       vector             map           sequence
```

MLP needs a fixed-length vector, so it uses 40 MFCC coefficient means plus
40 MFCC coefficient standard deviations.

CNN preserves the coefficient-by-time map so convolutional filters can learn local
patterns.

LSTM preserves temporal ordering so recurrent units can process the MFCC sequence over
time.


## 13. Reproducibility Checks


In [ ]:
# Purpose: Records the central data-foundation checks that downstream notebooks should
# depend on instead of duplicating dataset and MFCC logic.
reproducibility_checks = {
    "canonical_manifest_source": str(MANIFESTS_DIR),
    "shared_raw_mfcc_cache": str(MFCC_CACHE_DIR),
    "shared_class_weight_path": str(CLASS_WEIGHT_PATH),
    "model_notebooks_rescan_dataset": False,
    "model_notebooks_create_split": False,
    "model_notebooks_call_librosa_mfcc": False,
    "mfcc_walkthrough_location": "analysis notebook only",
    "mlp_representation": "40 means + 40 standard deviations = 80-D",
    "cnn_representation": "40 x T x 1 MFCC map",
    "lstm_representation": "T x 40 MFCC sequence",
    "full_mfcc_extraction_run_flag": RUN_FULL_SHARED_MFCC_EXTRACTION,
}
display(pd.DataFrame(list(reproducibility_checks.items()), columns=["check", "value"]))
